# Exploratory analysis of IBM HI-Small

**Full ledger, derived account roles, and the saved candidate scan.** This notebook explains the data and examines why the current business → business → person rule selected 905 chains but none of the 5,177 laundering-labeled transactions.

The reference label identifies synthetic money laundering. It does not identify confirmed embezzlement, employee ownership, a shell company, or a false invoice. Labels are used here for retrospective exploration, never to select or rank the saved candidates.

Run cells from top to bottom with this repository's `.venv` Python kernel. Install the optional analysis dependencies with `python -m pip install -e '.[eda]'`. The full dataset and prior scan outputs must already exist; this notebook does not download, import, or rerun detection. See `notebooks/README.md` for setup and input paths.

In [ ]:
import os
import sys
import json
import sqlite3
import hashlib
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def find_repository():
    override = os.environ.get('EMBEZZLEMENT_REPO_ROOT')
    candidates = [Path(override).expanduser()] if override else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'embezzlement_analysis').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Open this notebook inside the repository, or set EMBEZZLEMENT_REPO_ROOT.')

ROOT = find_repository()
DB_PATH = ROOT / 'data/hi-small.sqlite'
REPORT_DIR = ROOT / 'reports'
RAW_DIR = ROOT / 'data/raw/full-hi-small'
FIGURE_DIR = REPORT_DIR / 'eda/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required = [DB_PATH, REPORT_DIR / 'hi-small-audit.json', REPORT_DIR / 'hi-small-scan.json',
            REPORT_DIR / 'hi-small-candidates.jsonl', REPORT_DIR / 'hi-small-run-provenance.json',
            RAW_DIR / 'acquisition.json']
for source_name in ['HI-Small_Trans.csv', 'HI-Small_accounts.csv', 'HI-Small_Patterns.txt']:
    required.extend([RAW_DIR / source_name, RAW_DIR / (source_name + '.source.json')])
missing = [str(path.relative_to(ROOT)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing full-run inputs: ' + ', '.join(missing) + '. See notebooks/README.md.')

AUDIT = json.loads((REPORT_DIR / 'hi-small-audit.json').read_text())
SCAN = json.loads((REPORT_DIR / 'hi-small-scan.json').read_text())
ACQUISITION = json.loads((RAW_DIR / 'acquisition.json').read_text())
verification_path = REPORT_DIR / 'hi-small-final-verification.json'
VERIFY = json.loads(verification_path.read_text()) if verification_path.exists() else {}
conn = sqlite3.connect(DB_PATH.resolve().as_uri() + '?mode=ro', uri=True)
conn.execute('PRAGMA query_only=ON')
# One consistent read snapshot across all EDA queries. No database writes occur.
conn.execute('BEGIN')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)

COLORS = {'highlight': '#86BC25', 'green': '#046A38', 'blue': '#0076A8',
          'teal': '#00ABAB', 'gray': '#BBBCBC', 'text': '#000000',
          'secondary': '#53565A', 'grid': '#BBBCBC', 'white': '#FFFFFF'}
plt.rcParams.update({
    'figure.facecolor': COLORS['white'], 'axes.facecolor': COLORS['white'],
    'savefig.facecolor': COLORS['white'], 'font.family': 'DejaVu Sans',
    'font.size': 10, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'text.color': COLORS['text'], 'axes.labelcolor': COLORS['text'],
    'xtick.color': COLORS['text'], 'ytick.color': COLORS['text'],
    'axes.edgecolor': COLORS['grid'], 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.color': COLORS['grid'],
    'grid.linewidth': 0.5, 'figure.dpi': 110,
})

def show_table(frame, formats=None):
    view = frame.copy()
    if not isinstance(view.index, pd.RangeIndex):
        view = view.reset_index()
    formatters = {}
    for column in view.columns:
        if pd.api.types.is_integer_dtype(view[column]):
            formatters[column] = '{:,.0f}'
        elif pd.api.types.is_float_dtype(view[column]):
            formatters[column] = '{:,.3f}'
    formatters.update(formats or {})
    styled = view.style.format(formatters, na_rep='Not available').hide(axis='index')
    styled = styled.set_table_styles([
        {'selector': 'th', 'props': [('background-color', COLORS['text']), ('color', COLORS['white']),
                                     ('text-align', 'left'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '7px 9px'), ('border-bottom', '1px solid ' + COLORS['grid']),
                                     ('white-space', 'normal'), ('overflow-wrap', 'anywhere')]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('font-size', '12px')]},
    ])
    numeric = view.select_dtypes(include='number').columns
    if len(numeric):
        styled = styled.set_properties(subset=list(numeric), **{'text-align': 'right'})
    display(styled)

def finish_figure(fig, name):
    if not fig.get_constrained_layout():
        fig.tight_layout()
    fig.savefig(FIGURE_DIR / (name + '.png'), dpi=150, bbox_inches='tight', pad_inches=0.2)
    display(fig)
    plt.close(fig)

environment = pd.DataFrame([
    {'Component': 'Python', 'Version': platform.python_version()},
    *[{'Component': package, 'Version': version(package)} for package in
      ['pandas', 'numpy', 'matplotlib', 'nbformat', 'nbclient', 'ipykernel']],
])
show_table(environment)
display(Markdown('Database access is read-only. Full-ledger counts use SQL aggregation; only a bounded USD reference and the saved candidates enter pandas. Source amounts remain text/Decimal; plotting conversions are labeled.'))

## Source integrity and analysis scope

This notebook examines the complete **HI-Small v8** transaction snapshot, account metadata and published laundering-attempt annotations. The saved scan applies a predefined two-payment **business → business → person** rule. It produces review leads, not findings of embezzlement.

The next cell recomputes ledger row/label counts from SQLite and checks the audit, scan and acquisition manifests. It hashes the small audit/scan reports. **Raw-file SHA-256 values are inherited from completed acquisition and verification; this notebook does not rehash the 475 MB transaction CSV or account CSV by default.** Agreement among manifests confirms consistency of saved records, not that raw bytes could never have changed.

In [ ]:
import hashlib
import json
context_raw_dir = ROOT / 'data' / 'raw' / 'full-hi-small'
context_acquisition = json.loads((context_raw_dir / 'acquisition.json').read_text())
context_run = json.loads((REPORT_DIR / 'hi-small-run-provenance.json').read_text())
assert context_acquisition['status'] == 'complete_verified'
context_acquisition_by_name = {r['file_name']: r for r in context_acquisition['files']}
for context_name in ['hi-small-audit.json', 'hi-small-scan.json']:
    assert hashlib.sha256((REPORT_DIR / context_name).read_bytes()).hexdigest() == context_run['artifacts'][context_name]['sha256']
context_db_counts = pd.read_sql_query("""
    SELECT COUNT(*) AS rows, COALESCE(SUM(is_laundering=0),0) AS label_0,
           COALESCE(SUM(is_laundering=1),0) AS label_1 FROM transactions
""", conn).iloc[0]
assert int(context_db_counts['rows']) == AUDIT['transactions']['rows'] == SCAN['coverage']['transaction_count']
for context_label in ['0', '1']:
    assert int(context_db_counts['label_'+context_label]) == AUDIT['transactions']['label_counts'][context_label]
assert int(context_db_counts['label_0'] + context_db_counts['label_1']) == int(context_db_counts['rows'])
assert int(context_db_counts['label_1']) == SCAN['ledger_label_diagnostics']['laundering']
context_db_sources = pd.read_sql_query('SELECT file_name,sha256,row_count FROM sources ORDER BY sha256',conn)
assert len(context_db_sources) == 1, 'Expected the separate full HI-Small database.'
assert context_db_sources.iloc[0]['sha256'] == AUDIT['sources']['transactions']['sha256'] == SCAN['sources']['transactions'][0]['sha256']
assert int(context_db_sources.iloc[0]['row_count']) == int(context_db_counts['rows'])
assert SCAN['sources']['accounts']['sha256'] == AUDIT['sources']['accounts']['sha256']
context_record_counts = {'transactions':AUDIT['transactions']['rows'], 'accounts':AUDIT['accounts']['rows'],
                         'patterns':AUDIT['patterns']['transaction_rows']}
context_source_rows = []
for context_kind, context_source in AUDIT['sources'].items():
    context_manifest = context_acquisition_by_name[context_source['file_name']]
    context_sidecar = json.loads((context_raw_dir / (context_source['file_name']+'.source.json')).read_text())
    assert context_manifest['sha256'] == context_sidecar['sha256'] == context_source['sha256']
    assert context_manifest['status'] == context_sidecar['status'] == 'complete_verified'
    assert context_manifest['size_bytes'] == context_sidecar['size_bytes'] == context_source['bytes']
    assert (context_raw_dir / context_source['file_name']).stat().st_size == context_source['bytes']
    context_source_rows.append({'Source':context_source['file_name'], 'Records':context_record_counts[context_kind],
                                'Bytes':context_source['bytes'], 'SHA-256 (previously verified)':context_source['sha256']})
show_table(pd.DataFrame(context_source_rows), formats={'Records':'{:,.0f}', 'Bytes':'{:,.0f}'})
display(Markdown(f"**Fresh SQLite check passed:** {int(context_db_counts['rows']):,} rows; "
                 f"{int(context_db_counts['label_1']):,} labelled laundering; "
                 f"{int(context_db_counts['label_0']):,} labelled non-laundering. "
                 "Pattern records exclude BEGIN/END markers. Raw hashes were not recomputed here."))

## Account roles and source coverage

Synthetic name prefixes determine derived roles: `Corporation`, `Partnership` and `Sole Proprietorship` become **business**, `Individual` becomes **person**, and other names remain **unknown**. These account classifications do not establish legal form, employment, or shell status. Multiple accounts can belong to one entity.

Counts below reuse the independent account-file audit. Complete endpoint-to-metadata matching does **not** mean that every endpoint has a known role. The chart uses a logarithmic count axis to expose the small person/unknown groups; use printed counts and shares rather than interpreting bar lengths as linear proportions.

In [ ]:
context_role_counts = AUDIT['accounts']['row_role_counts_derived_from_name']
context_roles = pd.DataFrame([{'Derived role':r, 'Accounts':context_role_counts.get(r,0),
                              'Share of account rows':context_role_counts.get(r,0)/AUDIT['accounts']['rows']}
                             for r in ['business','unknown','person']])
assert int(context_roles['Accounts'].sum()) == AUDIT['accounts']['rows']
assert context_role_counts == SCAN['coverage']['account_roles']
show_table(context_roles, formats={'Accounts':'{:,.0f}', 'Share of account rows':'{:.3%}'})
context_fig, context_ax = plt.subplots(figsize=(8,2.8))
context_ax.barh(context_roles['Derived role'], context_roles['Accounts'],
                color=[COLORS.get('green','#86BC25'),COLORS.get('gray','#6B7280'),COLORS.get('blue','#0076A8')])
context_ax.invert_yaxis()
context_ax.set_xscale('log')
context_ax.set_xlim(1,context_roles['Accounts'].max()*3)
context_ax.set_xlabel('Account count — logarithmic scale')
context_ax.set_title('Very few account rows are classified as person')
for context_y,context_count in enumerate(context_roles['Accounts']):
    context_ax.text(context_count*1.1,context_y,f'{context_count:,}',va='center')
context_ax.grid(axis='x',alpha=0.15)
finish_figure(context_fig,'account-role-coverage')
context_n = AUDIT['transactions']['rows']
context_quality = pd.DataFrame([
    {'Measure':'Endpoint occurrences matched to unambiguous metadata',
     'Count':AUDIT['transactions']['endpoint_occurrence_coverage']['unambiguous_metadata'],
     'Denominator':2*context_n, 'Meaning':'Sender plus recipient occurrences; accounts repeat'},
    {'Measure':'Endpoint occurrences with unknown derived role',
     'Count':AUDIT['transactions']['endpoint_occurrence_roles_derived_from_name']['unknown'],
     'Denominator':2*context_n, 'Meaning':'Metadata exists; role-based selection excludes these endpoints'},
    {'Measure':'Self-transfers by exact bank/account identity',
     'Count':AUDIT['transactions']['self_transfer_rows'], 'Denominator':context_n,
     'Meaning':'Observable structure, not an independent fraud indicator'},
    {'Measure':'Adjacent timestamp reversals in original CSV order',
     'Count':AUDIT['transactions']['adjacent_timestamp_reversals'], 'Denominator':context_n-1,
     'Meaning':'File order is not chronology'}])
context_quality['Share'] = context_quality['Count']/context_quality['Denominator']
show_table(context_quality, formats={'Count':'{:,.0f}','Denominator':'{:,.0f}','Share':'{:.2%}'})
display(Markdown('The audit reports **740 person accounts**, **6,759 unknown-role accounts**, and '
                 '**2,418,736 adjacent timestamp reversals**. The detector orders connected transfers '
                 'by timestamps, not file position. Neither reversals nor self-transfers establish misconduct.'))

## Publisher main window versus the observed tail

The saved publisher description distinguishes the main period ending **2022-09-10** from later transactions and says the later records are all laundering. We retain that claim as source context and compare it with actual v8 labels. The following table aggregates the independent CSV audit's daily counts; the daily-profile section separately queries SQLite. These retrospective groups are not a train/test split.

In [ ]:
context_cutoff_date = AUDIT['transactions']['publisher_main_period_end']
context_days = pd.DataFrame(AUDIT['transactions']['daily_labels'])
context_days['Period'] = np.where(context_days['date']<=context_cutoff_date,
                                 'Main window: Sep 1–10','Tail: Sep 11–18')
context_periods = context_days.groupby('Period',sort=False)[['rows','label_0','label_1']].sum().reset_index()
context_periods['Laundering-label share'] = context_periods['label_1']/context_periods['rows']
assert int(context_periods['rows'].sum()) == int(context_db_counts['rows'])
assert int(context_periods['label_1'].sum()) == int(context_db_counts['label_1'])
context_tail = context_periods.loc[context_periods['Period'].str.startswith('Tail')].iloc[0]
assert int(context_tail['rows']) == AUDIT['transactions']['after_publisher_main_window']['rows']
show_table(context_periods.rename(columns={'rows':'Transactions','label_0':'Label 0','label_1':'Label 1'}),
           formats={'Transactions':'{:,.0f}','Label 0':'{:,.0f}','Label 1':'{:,.0f}','Laundering-label share':'{:.3%}'})
display(Markdown(f"**Observed tail: {int(context_tail['label_1']):,} / {int(context_tail['rows']):,} "
                 f"rows carry label 1 ({context_tail['Laundering-label share']:.2%}), not 100%.** "
                 f"The remaining {int(context_tail['label_0']):,} rows carry label 0. "
                 "This sharp change in prevalence makes one pooled rate insufficient to describe the dataset. "
                 "No calendar rule overwrites or infers labels."))

## Time, categories and amounts in the full ledger

These cells recompute summaries from the read-only database, restricted to the transaction source identified by the independent audit. They load grouped results, not the full 5.08-million-row ledger. `Is Laundering` is a synthetic AML annotation, **not an embezzlement outcome**. Labels support retrospective exploration here; they do not select detection leads.

In [ ]:
SOURCE_SHA = AUDIT["sources"]["transactions"]["sha256"]
MAIN_CUTOFF = pd.Timestamp("2022-09-11")
daily = pd.read_sql_query(
    """SELECT substr(timestamp, 1, 10) AS date, COUNT(*) AS rows,
              SUM(is_laundering) AS label_1,
              SUM(CASE WHEN is_laundering = 0 THEN 1 ELSE 0 END) AS label_0
       FROM transactions WHERE source_sha256 = ?
       GROUP BY substr(timestamp, 1, 10) ORDER BY date""",
    conn, params=(SOURCE_SHA,),
)
audit_daily = pd.DataFrame(AUDIT["transactions"]["daily_labels"])
compare_columns = ["date", "rows", "label_0", "label_1"]
pd.testing.assert_frame_equal(
    daily[compare_columns].reset_index(drop=True),
    audit_daily[compare_columns].reset_index(drop=True), check_dtype=False,
)
assert int(daily["rows"].sum()) == AUDIT["transactions"]["rows"]
daily["date"] = pd.to_datetime(daily["date"])
daily["label_rate_pct"] = 100 * daily["label_1"] / daily["rows"]
daily["phase"] = np.where(daily["date"] < MAIN_CUTOFF, "Main window", "Tail")
phase_summary = daily.groupby("phase", sort=False)[["rows", "label_0", "label_1"]].sum()
phase_summary["Label rate (%)"] = 100 * phase_summary["label_1"] / phase_summary["rows"]
show_table(phase_summary.rename(columns={"rows":"Transactions", "label_0":"Label 0", "label_1":"Label 1"}))
display(Markdown("Daily SQL counts and labels match the independent CSV audit exactly. The September 11 boundary follows the publisher's stated September 1–10 main window; it is not fitted to the labels."))

In [ ]:
from matplotlib.ticker import PercentFormatter
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
axes[0].bar(daily["date"], daily["rows"], width=0.5, color=COLORS["gray"])
axes[0].set_yscale("log")
axes[0].set_ylabel("Transactions (log scale)")
axes[0].set_title("Volume drops sharply after the main window", loc="left")
axes[1].plot(daily["date"], daily["label_rate_pct"], color=COLORS["green"], linewidth=2)
axes[1].set_ylabel("Laundering label rate (%)")
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=100))
axes[1].set_ylim(bottom=0)
axes[1].set_title("The smaller tail has a much higher positive-label share", loc="left")
for ax in axes:
    ax.axvline(MAIN_CUTOFF, color=COLORS["secondary"], linestyle="--", linewidth=1, label="Main-window end / tail start")
    ax.grid(axis="y", color=COLORS["grid"], linewidth=0.5)
    ax.set_axisbelow(True)
axes[0].legend(frameon=False, loc="upper right")
axes[1].annotate(f"{daily['label_rate_pct'].iloc[-1]:.1f}%", (daily["date"].iloc[-1], daily["label_rate_pct"].iloc[-1]), xytext=(-8, 8), textcoords="offset points", ha="right", color=COLORS["text"])
from matplotlib.dates import DateFormatter
tick_positions = np.unique(np.linspace(0, len(daily) - 1, min(7, len(daily)), dtype=int))
axes[1].set_xticks(daily["date"].iloc[tick_positions])
axes[1].xaxis.set_major_formatter(DateFormatter("%d %b"))
axes[1].set_xlim(daily["date"].iloc[0] - pd.Timedelta(days=0.6), daily["date"].iloc[-1] + pd.Timedelta(days=0.6))
axes[1].tick_params(axis="x", labelrotation=0)
axes[1].set_xlabel("Transaction date in source clock; no timezone inferred")
finish_figure(fig, "eda-daily-volume-label-rate")
display(Markdown("The panels use different units: transaction counts on a logarithmic scale above, label percentage on a linear scale below. The tail contains both labels; do not treat its date as a perfect laundering indicator."))

## Currency and payment format

One SQL aggregation counts each `(paid currency, payment format, label)` group. The tables and plots below derive their margins from that result. **Counts do not add monetary values across currencies.** Category label rates describe this synthetic ledger; they are not causal effects or calibrated fraud risks.

In [ ]:
category_joint = pd.read_sql_query(
    """SELECT payment_currency, payment_format, is_laundering, COUNT(*) AS transactions
       FROM transactions WHERE source_sha256 = ?
       GROUP BY payment_currency, payment_format, is_laundering""",
    conn, params=(SOURCE_SHA,),
)
assert int(category_joint["transactions"].sum()) == int(daily["rows"].sum())
category_joint["positive_transactions"] = category_joint["transactions"] * category_joint["is_laundering"]
def category_margin(column):
    out = category_joint.groupby(column, as_index=False)[["transactions", "positive_transactions"]].sum()
    out["label_rate_pct"] = 100 * out["positive_transactions"] / out["transactions"]
    return out.sort_values(["transactions", column], ascending=[False, True]).reset_index(drop=True)
currency_profile = category_margin("payment_currency")
format_profile = category_margin("payment_format")
assert dict(zip(currency_profile["payment_currency"], currency_profile["transactions"])) == AUDIT["transactions"]["payment_currency_counts"]
assert dict(zip(format_profile["payment_format"], format_profile["transactions"])) == AUDIT["transactions"]["payment_format_counts"]
assert int(category_joint["positive_transactions"].sum()) == int(daily["label_1"].sum())

In [ ]:
display(Markdown("**Eight largest paid-currency groups by transaction count** (remaining currencies stay in `currency_profile`)."))
show_table(currency_profile.head(8).rename(columns={"payment_currency":"Paid currency", "transactions":"Transactions", "positive_transactions":"Label 1", "label_rate_pct":"Label rate (%)"}))
display(Markdown("**All payment formats**, ordered by transaction count."))
show_table(format_profile.rename(columns={"payment_format":"Payment format", "transactions":"Transactions", "positive_transactions":"Label 1", "label_rate_pct":"Label rate (%)"}))

In [ ]:
from matplotlib.ticker import FuncFormatter
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for row_axes, frame, key, label in [
    (axes[0], currency_profile.head(8), "payment_currency", "Eight largest paid currencies"),
    (axes[1], format_profile, "payment_format", "All payment formats"),
]:
    names = frame[key].tolist()
    y = np.arange(len(frame))
    row_axes[0].barh(y, frame["transactions"], height=0.5, color=COLORS["gray"])
    row_axes[1].barh(y, frame["label_rate_pct"], height=0.5, color=COLORS["green"])
    row_axes[0].set_title(label + ": volume", loc="left")
    row_axes[1].set_title(label + ": label share", loc="left")
    row_axes[0].set_xlabel("Transactions")
    row_axes[0].xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1_000_000:g}m"))
    row_axes[1].set_xlabel("Laundering label rate (%)")
    row_axes[1].xaxis.set_major_formatter(PercentFormatter(xmax=100))
    for ax in row_axes:
        ax.set_yticks(y)
        ax.set_yticklabels(names)
        ax.invert_yaxis()
        ax.grid(False)
        ax.set_xlim(left=0)
finish_figure(fig, "eda-category-volume-label-rate")

## Paid amounts in US dollars only

Load **all USD-paid label-1 rows**, plus USD-paid label-0 rows whose immutable database ID is divisible by 100. This is a deterministic systematic reference sample, **not a random sample**; source ordering may bias it. The inclusion rule deliberately changes class proportions, so do not estimate label prevalence from this frame.

Keep `amount_paid` as the original text and retain source hash and CSV record number. Exact decimal values support the descriptive table. Convert to floating point only for the logarithmic plot. No exchange-rate assumptions or cross-currency sums enter this section.

In [ ]:
from decimal import Decimal
USD_CURRENCY = "US Dollar"
REFERENCE_INTERVAL = 100
usd_reference = pd.read_sql_query(
    """SELECT id, source_sha256, source_row, amount_paid, is_laundering
       FROM transactions
       WHERE source_sha256 = ? AND payment_currency = ?
         AND (is_laundering = 1 OR (is_laundering = 0 AND id % ? = 0))
       ORDER BY id""",
    conn, params=(SOURCE_SHA, USD_CURRENCY, REFERENCE_INTERVAL),
)
assert usd_reference["amount_paid"].map(lambda value: isinstance(value, str)).all()
assert (usd_reference.loc[usd_reference["is_laundering"].eq(0), "id"] % REFERENCE_INTERVAL).eq(0).all()
usd_reference["amount_decimal"] = usd_reference["amount_paid"].map(Decimal)
assert usd_reference["amount_decimal"].map(lambda value: value.is_finite() and value > 0).all()
full_usd_counts = category_joint.loc[category_joint["payment_currency"].eq(USD_CURRENCY)].groupby("is_laundering")["transactions"].sum()
assert int(usd_reference["is_laundering"].eq(1).sum()) == int(full_usd_counts.get(1, 0))
assert len(usd_reference) < 100_000, "Unexpected reference size: revisit the notebook memory bound."

In [ ]:
usd_rows = []
for label, group in usd_reference.groupby("is_laundering", sort=True):
    values = sorted(group["amount_decimal"])
    middle = len(values) // 2
    median = values[middle] if len(values) % 2 else (values[middle - 1] + values[middle]) / Decimal(2)
    usd_rows.append({
        "Group": "Label 0: systematic reference" if label == 0 else "Label 1: all USD-paid rows",
        "Loaded rows": len(group), "Full USD group rows": int(full_usd_counts.get(label, 0)),
        "Minimum paid (USD)": str(values[0]), "Median paid (USD)": str(median), "Maximum paid (USD)": str(values[-1]),
    })
show_table(pd.DataFrame(usd_rows))
display(Markdown("Amount statistics describe the loaded rows only. The full-group count provides the denominator without implying the systematic reference is representative."))

In [ ]:
plot_amounts = usd_reference["amount_decimal"].map(float).to_numpy()
assert np.isfinite(plot_amounts).all() and (plot_amounts > 0).all()
log_amounts = np.log10(plot_amounts)
low, high = float(log_amounts.min()), float(log_amounts.max())
if low == high:
    low, high = low - 0.5, high + 0.5
bins = np.linspace(low, high, 36)
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for label, colour, legend in [
    (0, COLORS["highlight"], "Label 0: systematic reference"),
    (1, COLORS["green"], "Label 1: all USD-paid rows"),
]:
    selected = log_amounts[usd_reference["is_laundering"].to_numpy() == label]
    if len(selected):
        ax.hist(selected, bins=bins, weights=np.full(len(selected), 100.0 / len(selected)), histtype="step", linewidth=2, color=colour, label=f"{legend} (n={len(selected):,})")
ax.set_title("Compare USD-paid amount shapes within each label group", loc="left")
ax.set_xlabel("log₁₀(amount paid in USD); 2 = $100, 3 = $1,000, 4 = $10,000")
ax.set_ylabel("Share of each loaded group in bin (%)")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_ylim(bottom=0)
ax.grid(axis="y", color=COLORS["grid"], linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(frameon=False)
finish_figure(fig, "eda-usd-paid-amount-shapes")
display(Markdown("Each group's bin shares sum to 100%. The overlaid step histograms compare shape, not class prevalence, transaction totals, losses or stolen proceeds."))

### What this profile can and cannot establish

The fresh SQL summaries independently reproduce the audit's daily and category totals. They expose temporal imbalance, category composition and USD amount shapes. They do not validate ownership, employee relationships, shell-company status, invoice validity or embezzlement. The systematic amount reference can reflect source ordering; treat differences as exploratory observations, not a tested predictive result.

## Candidate-path analysis: overlap, timing, and reference-label coverage

This section reads the **completed scan**, not a new detection run. It examines every saved business → business → person candidate. The 905 paths overlap: they are **not 905 independent incidents**, and their transaction amounts must not be summed as losses. IBM name prefixes supply the entity roles; they do not establish employment, authorization, or shell-company status.

In [ ]:
import hashlib
import json
from decimal import Decimal, localcontext
from fractions import Fraction

candidate_path = REPORT_DIR / SCAN['candidate_output']['file_name']
candidate_bytes = candidate_path.read_bytes()
assert hashlib.sha256(candidate_bytes).hexdigest() == SCAN['candidate_output']['sha256'], 'Candidate JSONL hash mismatch'
assert len(candidate_bytes) == SCAN['candidate_output']['bytes'], 'Candidate JSONL size mismatch'
candidate_records = [json.loads(line) for line in candidate_bytes.splitlines() if line.strip()]
assert len(candidate_records) == SCAN['candidate_count'] == SCAN['candidate_output']['row_count'] == 905
assert SCAN['full_coverage'] is True and SCAN['truncated'] is False
transaction_sources = {source['sha256']: source for source in SCAN['sources']['transactions']}
account_source_hash = SCAN['sources']['accounts']['sha256']
for record in candidate_records:
    assert len(record['path']) == 2 and len(record['accounts']) == 3
    for edge in record['path']:
        assert edge['source_sha256'] in transaction_sources
        assert 2 <= edge['source_row'] <= transaction_sources[edge['source_sha256']]['row_count'] + 1
        assert 'is_laundering' not in edge, 'Labels must not be present in candidate selection records'
    assert all(account['source_sha256'] == account_source_hash for account in record['accounts'])
display(Markdown(f"**Verified saved output:** {len(candidate_records):,} paths; SHA-256 `{SCAN['candidate_output']['sha256']}`. No import or scan is rerun."))

In [ ]:
# Keep original amount strings and exact rational ratios. Decimal parses the
# amounts exactly; the 50-digit Decimal/float view below is only for display.
pair_rows = []
for record in candidate_records:
    first, second = record['path']
    origin, middle, person = record['accounts']
    ratio_exact = Fraction(Decimal(second['amount_paid'])) / Fraction(Decimal(first['amount_received']))
    lag_seconds = int((pd.Timestamp(second['timestamp']) - pd.Timestamp(first['timestamp'])).total_seconds())
    assert 0 < lag_seconds <= int(Decimal(SCAN['parameters']['window_hours']) * 3600)
    assert first['receiving_currency'] == second['payment_currency']
    assert Fraction(Decimal(SCAN['parameters']['min_ratio'])) <= ratio_exact <= Fraction(Decimal(SCAN['parameters']['max_ratio']))
    assert len({(a['bank'], a['account']) for a in record['accounts']}) == 3
    assert len({a['entity_id'] for a in record['accounts']}) == 3
    with localcontext() as decimal_context:
        decimal_context.prec = 50
        ratio_decimal = Decimal(ratio_exact.numerator) / Decimal(ratio_exact.denominator)
    pair_rows.append({
        'first_id': first['id'], 'second_id': second['id'],
        'first_source_sha256': first['source_sha256'], 'second_source_sha256': second['source_sha256'],
        'first_source_row': first['source_row'], 'second_source_row': second['source_row'],
        'first_source_ref': f"sha256:{first['source_sha256']}#record={first['source_row']}",
        'second_source_ref': f"sha256:{second['source_sha256']}#record={second['source_row']}",
        'first_timestamp': first['timestamp'], 'second_timestamp': second['timestamp'],
        'first_amount_paid_text': first['amount_paid'],
        'first_amount_received_text': first['amount_received'],
        'second_amount_paid_text': second['amount_paid'],
        'second_amount_received_text': second['amount_received'],
        'currency': first['receiving_currency'],
        'ratio_exact': ratio_exact, 'ratio_decimal': str(ratio_decimal), 'ratio': float(ratio_decimal),
        'lag_hours': lag_seconds / 3600,
        'origin_entity_id': origin['entity_id'], 'origin_entity_name': origin['entity_name'],
        'middle_entity_id': middle['entity_id'], 'middle_entity_name': middle['entity_name'],
        'person_entity_id': person['entity_id'], 'person_entity_name': person['entity_name'],
    })
pairs = pd.DataFrame(pair_rows)
assert not pairs.duplicated(['first_id', 'second_id']).any()
selected_ids = sorted(set(pairs['first_id']) | set(pairs['second_id']))
assert pairs['first_id'].nunique() == SCAN['unique_first_edges'] == 355
assert pairs['second_id'].nunique() == SCAN['unique_second_edges'] == 93
assert len(selected_ids) == SCAN['selected_edge_diagnostics']['total'] == 448
show_table(pd.DataFrame([
    {'Counting unit': 'Two-hop candidate paths', 'Count': len(pairs)},
    {'Counting unit': 'Distinct first-hop transactions', 'Count': pairs['first_id'].nunique()},
    {'Counting unit': 'Distinct second-hop transactions', 'Count': pairs['second_id'].nunique()},
    {'Counting unit': 'Distinct transactions in the union of both hops', 'Count': len(selected_ids)},
]))
funnel = pd.DataFrame({
    'Stage': ['Role, strict time, and currency checks', 'Inclusive adjacent-amount ratio check', 'Three distinct accounts and entities'],
    'Candidate pairs': [SCAN['examined']['time_currency_role_pairs'], SCAN['examined']['ratio_pass_pairs'], len(pairs)],
})
assert funnel['Candidate pairs'].tolist() == [19577, 932, 905]
assert SCAN['examined']['ownership_excluded_pairs'] == 27
show_table(funnel)
show_table(pd.DataFrame([
    {'Rule': 'Entity-role sequence', 'Configured value': 'business → business → person (derived IBM roles)'},
    {'Rule': 'Time order', 'Configured value': 'Second timestamp must be strictly later'},
    {'Rule': 'Total window', 'Configured value': SCAN['parameters']['window_hours'] + ' hours, inclusive upper bound'},
    {'Rule': 'Adjacent currency', 'Configured value': 'First received currency = second paid currency'},
    {'Rule': 'Amount ratio', 'Configured value': SCAN['parameters']['min_ratio'] + ' ≤ second paid / first received ≤ ' + SCAN['parameters']['max_ratio']},
    {'Rule': 'Identity', 'Configured value': 'Three distinct bank/account pairs and entity IDs'},
    {'Rule': 'Labels', 'Configured value': 'Excluded from selection and ranking'},
]))

The funnel counts **pairs**, not transactions: 19,577 pairs met the role, time, and currency conditions; 932 also met the amount ratio; 27 repeated an account or entity, leaving 905. These thresholds are engineering rules, not ACFE-prescribed proof of fraud. The amount ratio compares the first recipient's received amount with the next sender's paid amount **in the same currency**. We do not compare amounts across different currencies or attribute mixed funds to a particular origin.

In [ ]:
# Evaluate reference labels only after candidate IDs and all selection checks
# are fixed. Each read-only query binds at most 400 IDs.
label_chunks = []
for offset in range(0, len(selected_ids), 400):
    chunk = [int(value) for value in selected_ids[offset:offset + 400]]
    placeholders = ','.join('?' for _ in chunk)
    label_chunks.append(pd.read_sql_query(
        f'SELECT id,is_laundering FROM transactions WHERE id IN ({placeholders})',
        conn, params=chunk))
selected_labels = pd.concat(label_chunks, ignore_index=True)
assert len(selected_labels) == len(selected_ids) == 448
assert set(selected_labels['id']) == set(selected_ids)
assert selected_labels['id'].is_unique
assert set(selected_labels['is_laundering']) == {0}
label_by_id = selected_labels.set_index('id')['is_laundering']
pairs['first_reference_label'] = pairs['first_id'].map(label_by_id)
pairs['second_reference_label'] = pairs['second_id'].map(label_by_id)
assert int(selected_labels['is_laundering'].sum()) == SCAN['selected_edge_diagnostics']['laundering'] == 0
assert SCAN['ledger_label_diagnostics']['laundering'] == 5177
show_table(pd.DataFrame([
    {'Reference-label measure': 'Selected distinct transactions labeled laundering', 'Count': 0},
    {'Reference-label measure': 'All ledger transactions labeled laundering', 'Count': 5177},
    {'Reference-label measure': 'Selected distinct transactions labeled 0', 'Count': 448},
]))
display(Markdown('**0 of 5,177 laundering-labeled transactions appear in the selected edge union.** '
    'This is zero reference-label coverage for this narrow rule and indicates a poor fit to the dataset’s labeled laundering cases. '
    'It is **not accuracy**, proof that these candidates are benign, or an embezzlement evaluation: the dataset supplies no embezzlement ground truth. '
    'The label queries did not choose, remove, or rank any candidate.'))

In [ ]:
entity_counts = pd.DataFrame([
    {'Entity position': 'Origins', 'Distinct entities': pairs['origin_entity_id'].nunique()},
    {'Entity position': 'Intermediaries', 'Distinct entities': pairs['middle_entity_id'].nunique()},
    {'Entity position': 'Person recipients', 'Distinct entities': pairs['person_entity_id'].nunique()},
])
show_table(entity_counts)
recipient_concentration = (
    pairs.groupby(['person_entity_id', 'person_entity_name'], sort=True)
    .agg(candidate_paths=('first_id', 'size'), distinct_second_hops=('second_id', 'nunique'))
    .reset_index()
    .sort_values(['candidate_paths', 'person_entity_id'], ascending=[False, True], kind='stable')
)
recipient_concentration['share_of_paths_pct'] = recipient_concentration['candidate_paths'] / len(pairs) * 100
recipient_concentration['paths_per_second_hop'] = recipient_concentration['candidate_paths'] / recipient_concentration['distinct_second_hops']
assert int(recipient_concentration['candidate_paths'].sum()) == 905
assert int(recipient_concentration['distinct_second_hops'].sum()) == 93
show_table(recipient_concentration.head(8).rename(columns={
    'person_entity_id': 'Recipient entity ID', 'person_entity_name': 'Recipient name',
    'candidate_paths': 'Candidate paths (count)', 'distinct_second_hops': 'Distinct onward transactions (count)',
    'share_of_paths_pct': 'Share of paths (%)', 'paths_per_second_hop': 'Paths per onward transaction',
}).round({'Share of paths (%)': 2, 'Paths per onward transaction': 2}))

The charts expose two different properties of the saved candidates. The scatter shows the configured timing and amount-ratio region. The recipient comparison separates **path count** from **distinct onward-transaction count**: one outward payment can appear in several paths when several earlier inflows qualify. High path concentration therefore does not, by itself, measure the number or value of independent events. Recipients are ordered by path count with entity ID as the tie-breaker; labels play no part.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.4))
ax.scatter(pairs['lag_hours'], pairs['ratio'], s=24, color=COLORS['blue'], alpha=0.45, linewidths=0)
ax.set_title('Timing and amount ratios across 905 candidate paths', loc='left', color=COLORS['text'])
ax.set_xlabel('Elapsed time between hops (hours)')
ax.set_ylabel('Second paid amount / first received amount (ratio)')
ax.set_xlim(0, float(Decimal(SCAN['parameters']['window_hours'])))
ax.set_ylim(float(Decimal(SCAN['parameters']['min_ratio'])) - 0.01,
            float(Decimal(SCAN['parameters']['max_ratio'])) + 0.01)
ax.grid(axis='both', color=COLORS['grid'], linewidth=0.6, alpha=0.6)
ax.set_axisbelow(True)
for boundary in ['min_ratio', 'max_ratio']:
    ax.axhline(float(Decimal(SCAN['parameters'][boundary])), color=COLORS['gray'], linestyle='--', linewidth=0.8)
fig.text(0.01, 0.01, 'Each point represents a path. Paths can share transactions; overlapping points are not jittered.',
         color=COLORS['secondary'], fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 1))
finish_figure(fig, 'candidate_lag_ratio')

In [ ]:
recipient_top = recipient_concentration.head(8).copy()
y = np.arange(len(recipient_top))
fig, ax = plt.subplots(figsize=(10, 6))
path_bars = ax.barh(y - 0.18, recipient_top['candidate_paths'], height=0.34,
                   color=COLORS['green'], label='Candidate paths (count)')
edge_bars = ax.barh(y + 0.18, recipient_top['distinct_second_hops'], height=0.34,
                   color=COLORS['blue'], label='Distinct onward transactions (count)')
ax.set_yticks(y)
ax.set_yticklabels(recipient_top['person_entity_name'] + ' · ' + recipient_top['person_entity_id'])
ax.invert_yaxis()
ax.set_xlabel('Count — different counting units; do not add the two bars')
ax.set_title('Recipient concentration: paths versus distinct onward transactions', loc='left', color=COLORS['text'])
ax.set_xlim(0, max(recipient_top['candidate_paths']) * 1.18)
ax.grid(axis='x', color=COLORS['grid'], linewidth=0.6, alpha=0.6)
ax.set_axisbelow(True)
for bars in [path_bars, edge_bars]:
    for bar in bars:
        ax.text(bar.get_width() + max(recipient_top['candidate_paths']) * 0.015,
                bar.get_y() + bar.get_height() / 2, f'{int(bar.get_width()):,}',
                va='center', fontsize=9, color=COLORS['text'])
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
finish_figure(fig, 'recipient_concentration')

In [ ]:
# Sample by source provenance order, never by target label or suspiciousness.
sample = pairs.sort_values(
    ['first_source_sha256', 'first_source_row', 'second_source_sha256', 'second_source_row'],
    kind='stable').head(5).copy()
sample_table = pd.DataFrame({
    'First source record': [transaction_sources[h]['file_name'] + '#record=' + str(r)
                            for h, r in zip(sample['first_source_sha256'], sample['first_source_row'])],
    'Second source record': [transaction_sources[h]['file_name'] + '#record=' + str(r)
                             for h, r in zip(sample['second_source_sha256'], sample['second_source_row'])],
    'Person recipient': sample['person_entity_name'].to_numpy(),
    'Lag (hours)': sample['lag_hours'].round(2).to_numpy(),
    'Amount ratio': sample['ratio'].round(5).to_numpy(),
    'Adjacent currency': sample['currency'].to_numpy(),
    'Reference labels (first / second)': [f'{a} / {b}' for a, b in
                                         zip(sample['first_reference_label'], sample['second_reference_label'])],
})
show_table(sample_table)
display(Markdown('Source record numbers include the CSV header as record 1. Full SHA-256 references and '
    'original amount strings remain in `pairs`; the manifest above identifies the source files. '
    '**Label 0 is a dataset reference label, not a finding of innocence. These 905 overlapping paths are review leads, '
    'not independent incidents, proven embezzlement, or calculated losses.**'))

## Does the annotated-pattern subset contain the target shape?

This separate retrospective diagnostic checks two-payment chains **within each labelled laundering attempt**, using the same 72-hour, currency, ratio, distinct-account and distinct-entity conditions. Pattern annotations enter this evaluation only; they never enter candidate selection. This subset cannot measure full-ledger recall because it omits unannotated and cross-attempt chains.

In [ ]:
context_pattern_path = REPORT_DIR / 'hi-small-pattern-scope.json'
if not context_pattern_path.exists():
    display(Markdown('The optional pattern-scope report is unavailable. Within-attempt scope checks were not run in this notebook; no zero-match result is inferred.'))
else:
    context_pattern_bytes = (REPORT_DIR/'hi-small-pattern-scope.json').read_bytes()
    context_pattern_artifact = context_run.get('artifacts', {}).get('hi-small-pattern-scope.json')
    if context_pattern_artifact is not None:
        assert hashlib.sha256(context_pattern_bytes).hexdigest() == context_pattern_artifact['sha256']
    context_patterns = json.loads(context_pattern_bytes)
    for context_source in context_patterns['sources'].values():
        assert context_source['sha256'] == context_acquisition_by_name[context_source['file_name']]['sha256']
    context_pattern_table = pd.DataFrame([
        {'Pattern type':name, 'Attempts':v['attempt_count'], 'Annotated rows':v['pattern_row_count'],
         'Eligible pairs':v['eligible_pair_count'], 'Matched attempts':v['matched_attempt_count']}
        for name,v in context_patterns['by_pattern_type'].items()])
    assert int(context_pattern_table['Attempts'].sum()) == AUDIT['patterns']['attempts']
    assert int(context_pattern_table['Annotated rows'].sum()) == AUDIT['patterns']['transaction_rows']
    show_table(context_pattern_table, formats={n:'{:,.0f}' for n in
                                             ['Attempts','Annotated rows','Eligible pairs','Matched attempts']})
    context_pattern_rows = context_patterns['scope']['pattern_row_count']
    context_pattern_persons = context_patterns['scope']['pattern_account_roles'].get('person',0)
    display(Markdown(f"The {context_patterns['scope']['attempt_count']:,} attempts contain "
                     f"{context_pattern_rows:,} annotated rows and **{context_pattern_persons:,} person-classified accounts**. "
                     f"Of {context_patterns['funnel']['connected_time_window_pairs']:,} connected pairs within the time window, "
                     f"{context_patterns['eligible_pair_count']:,} meet the full role-and-payment rule. "
                     "The business → business → person hypothesis does not occur in this annotated subset. "
                     "Zero matches is a scope mismatch, not proof that the scanner missed every laundering pattern.\n\n"
                     f"The pattern-row count ({context_pattern_rows:,}) is smaller than the ledger's "
                     f"{int(context_db_counts['label_1']):,} labelled-laundering rows: a count gap of "
                     f"{int(context_db_counts['label_1'])-context_pattern_rows:,}. "
                     "The audit found every published pattern signature in the ledger, but this diagnostic "
                     "does not establish one-to-one attribution for every labelled ledger row."))

## What the data can support

The selected paths describe a payment shape. Neither a path nor a simulator label establishes embezzlement, a shell vendor, employee involvement, a false invoice, or an unauthorized payment. Thresholds were fixed before evaluating labels. Label overlap is descriptive; it is not validated accuracy for the user's embezzlement scenario.

| ACFE-informed check | Evidence needed | HI-Small status |
|---|---|---|
| Employee/vendor links | Employee/vendor records and verified account or ownership links | **Not run — source data unavailable** |
| Invoice / purchase-order / receipt consistency | Invoices, orders and delivery or acceptance evidence | **Not run — source data unavailable** |
| Purchase/payment approval separation | Approver identities, authority and exceptions | **Not run — source data unavailable** |
| Shell-company or beneficial-owner assessment | Filings, operational evidence and corroborated ownership | **Not established by synthetic names** |
| Embezzlement case evaluation | Independently reviewed case outcomes | **Not computable — no embezzlement case labels** |

**Interpretation limits:** Roles come from name prefixes and lack effective dates. Missing/unknown roles narrow the observable hypothesis. Minute-resolution timestamps exclude equal-time hops. The 72-hour/0.80–1.05 filters are engineering choices, not ACFE thresholds. The adjacent intermediary currencies must match, but either transfer may itself convert currencies. This two-hop rule does not attribute value across mismatched intermediary currencies or reconstruct longer paths and split/merged flows. Similar amounts do not prove that the same money moved. Overlapping paths are not independent observations: their count is not a case count, and their amounts must not be summed into losses. Consider legitimate explanations and evidence that weakens an allegation.

**Sources:** [IBM dataset on Kaggle](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml), [IBM AML-Data repository](https://github.com/IBM/AML-Data), [IBM dataset paper](https://arxiv.org/html/2306.16424v3), [ACFE Anti-Fraud Data Analytics Tests](https://www.acfe.com/fraud-resources/fraud-risk-tools---coso/anti-fraud-data-analytics-tests), [ACFE Shell Companies](https://www.acfe.com/fraud-resources/shell-companies), and [CFE professional standards and interpretation](https://www.acfe.com/about-the-acfe/-/media/9560BECA7A1441B5B3CD9A6E14CE7B89.ashx). Retrieval records and hashes remain with the local analysis.

## Reproducibility and next experiments

The headline counts use the complete ledger. The amount-distribution reference is deliberately smaller and cannot estimate prevalence. Candidate concentration describes overlapping paths, not independent fraud incidents or calculated losses.

Useful next experiments require independently labeled company/employee relationships, authorized invoices and approvals, and legitimate supplier-payroll lookalikes. Compare predefined alternatives using chronological evaluation and report missing coverage. Keep AML labels and pattern membership outside feature construction and candidate selection.

This notebook saves its figures under `reports/eda/figures/`. The source notebook keeps outputs cleared in Git; the executed copy and HTML preview stay under `reports/eda/` with the other local analysis outputs.

In [ ]:
assert SCAN['full_coverage'] is True and SCAN['truncated'] is False
assert len(pairs) == SCAN['candidate_count']
assert len(selected_ids) == SCAN['selected_edge_diagnostics']['total']
conn.rollback()
conn.close()
display(Markdown('EDA completed. All source reconciliation and saved-candidate checks passed; the database connection is closed.'))